# Exercise

## Goal
The purpose of this notebook is to build a smart cookbook agent.

The user can provide a list of ingredients and the number of meals to cook to the agent.

On that basis the agent will propose 3 recipes to the user, making sure he uses as much available ingredients as possible. The agent also provides the list of additional ingredients required and the cooking steps.

In [24]:
# Start with some imports - rich is a library for making formatted text output in the terminal

import copy
import json
import re

from dotenv import load_dotenv
from openai import OpenAI
from rich.console import Console

load_dotenv(override=True)

True

In [25]:
def show(text):
    try:
        Console().print(text)
    except Exception:
        print(text)

In [26]:
openai = OpenAI()

In [27]:
# Persistent lists

recipes = []  # list of recipes.
# Each recipe is a dictionary with the following keys:
# - name: the name of the recipe
# - servings: the number of servings the recipe makes (int)
# - ingredients: list of dicts: name (str), quantity (float, required amount for current servings), unit (str)
# - steps: list of strings

available_ingredients = []  # Pantry after the last set_available_ingredients call.
# Each item: name, quantity (float), unit (str).
# Reconcile does not read this list; it uses _pantry_original. Cleared/rebuilt when set_available_ingredients runs.

available_ingredients_used = []  # Filled by reconcile_recipes_with_pantry: pantry amount applied to each recipe line (min(need, have)).
# Each item: recipe (str), name, quantity (float), unit (str).

additional_ingredients_required = []  # Filled by reconcile: shortfall vs pantry for a matched line, or full need if no pantry line matched (same name+unit after normalization).
# Each item: recipe (str), name, quantity (float), unit (str).

_pantry_original = [] # Deep copy of available_ingredients taken inside set_available_ingredients. Reconcile deep-copies this per recipe so each recipe is compared to the full original pantry.


In [28]:
# Helper functions

def _norm_name(name):
    return re.sub(r"\s+", " ", str(name).strip().lower())


def _norm_unit(unit):
    return str(unit).strip().lower()


In [29]:
# Tool functions

def set_available_ingredients(items):
    """Replace pantry, snapshot for reconciliation, clear prior run state."""
    global _pantry_original
    recipes.clear()
    available_ingredients.clear()
    available_ingredients_used.clear()
    additional_ingredients_required.clear()
    for item in items:
        row = {
            "name": item["name"],
            "quantity": float(item["quantity"]),
            "unit": item["unit"],
        }
        available_ingredients.append(row)
    _pantry_original = copy.deepcopy(available_ingredients)
    show(f"PANTRY SET: {len(available_ingredients)} items")
    return {"items": len(available_ingredients), "pantry": available_ingredients}


def create_recipes_list(recipe_items):
    """Add multiple recipes from one tool call (parameter name matches tool schema)."""
    result = ""
    added = []
    for index, r in enumerate(recipe_items):
        recipe = {
            "name": r["name"],
            "servings": int(r["servings"]),
            "ingredients": [
                {
                    "name": x["name"],
                    "quantity": float(x["quantity"]),
                    "unit": x["unit"],
                }
                for x in r["ingredients"]
            ],
            "steps": list(r["steps"]),
        }
        recipes.append(recipe)
        added.append(recipe)
        nm = recipe["name"]
        sv = recipe["servings"]
        result += f"Recipe #{index + 1}: {nm} - {sv} servings\n"
    show("CREATED RECIPES")
    show(result)
    return added


def adjust_recipe_servings(recipe_name, desired_servings):
    """Scale one recipe's ingredient quantities and servings."""
    for recipe in recipes:
        if recipe["name"] == recipe_name:
            original_servings = recipe["servings"]
            if original_servings == 0:
                raise ValueError("Original servings cannot be zero.")
            scale = desired_servings / original_servings
            for ing in recipe["ingredients"]:
                ing["quantity"] = ing["quantity"] * scale
            recipe["servings"] = int(desired_servings)
            show(f"ADJUSTED SERVINGS FOR {recipe_name} from {original_servings} to {desired_servings}")
            return recipe
    show(f"Recipe '{recipe_name}' not found.")
    return None


def adjust_all_recipes_servings(desired_servings):
    """Scale every recipe in recipes[] to the same number of servings (one tool call)."""
    adjusted = []
    for recipe in list(recipes):
        r = adjust_recipe_servings(recipe["name"], desired_servings)
        if r:
            adjusted.append(r["name"])
    return {"adjusted": adjusted, "desired_servings": desired_servings}


def reconcile_recipes_with_pantry(recipe_names=None):
    """
    For each recipe, compare scaled ingredients to a fresh copy of the original pantry.
    Fills available_ingredients_used and additional_ingredients_required (per recipe).
    """
    if not _pantry_original:
        msg = "Pantry empty: call set_available_ingredients first."
        show(msg)
        return {"error": msg}

    available_ingredients_used.clear()
    additional_ingredients_required.clear()

    targets = recipes
    if recipe_names:
        names = set(recipe_names)
        targets = [r for r in recipes if r["name"] in names]

    summary_lines = []
    for recipe in targets:
        pantry_work = copy.deepcopy(_pantry_original)
        for ing in recipe["ingredients"]:
            need = float(ing["quantity"])
            n_name = _norm_name(ing["name"])
            n_unit = _norm_unit(ing["unit"])
            match_idx = None
            for i, p in enumerate(pantry_work):
                if _norm_name(p["name"]) == n_name and _norm_unit(p["unit"]) == n_unit:
                    match_idx = i
                    break
            if match_idx is not None:
                have = float(pantry_work[match_idx]["quantity"])
                take = min(need, have)
                if take > 0:
                    available_ingredients_used.append(
                        {
                            "recipe": recipe["name"],
                            "name": ing["name"],
                            "quantity": take,
                            "unit": ing["unit"],
                        }
                    )
                    pantry_work[match_idx]["quantity"] = have - take
                shortfall = need - take
                if shortfall > 0:
                    additional_ingredients_required.append(
                        {
                            "recipe": recipe["name"],
                            "name": ing["name"],
                            "quantity": shortfall,
                            "unit": ing["unit"],
                        }
                    )
            else:
                additional_ingredients_required.append(
                    {
                        "recipe": recipe["name"],
                        "name": ing["name"],
                        "quantity": need,
                        "unit": ing["unit"],
                    }
                )
        summary_lines.append(recipe["name"])

    show("RECONCILED PANTRY VS RECIPES")
    show(
        f"used entries: {len(available_ingredients_used)}, "
        f"additional needed entries: {len(additional_ingredients_required)}"
    )
    return {
        "recipes_reconciled": summary_lines,
        "used": available_ingredients_used,
        "additional_required": additional_ingredients_required,
    }

In [30]:
# Tools

set_available_ingredients_json = {
    "name": "set_available_ingredients",
    "description": "Set the kitchen pantry from structured data. Call this first. Clears prior recipes and reconciliation lists; resets the pantry snapshot used for shopping-list math.",
    "parameters": {
        "type": "object",
        "properties": {
            "items": {
                "type": "array",
                "items": {
                    "type": "object",
                    "properties": {
                        "name": {"type": "string"},
                        "quantity": {"type": "number"},
                        "unit": {"type": "string"},
                    },
                    "required": ["name", "quantity", "unit"],
                    "additionalProperties": False,
                },
            }
        },
        "required": ["items"],
        "additionalProperties": False,
    },
}

create_recipes_list_json = {
    "name": "create_recipes_list",
    "description": "Add new recipes to the recipes[] list",
    "parameters": {
        "type": "object",
        "properties": {
            "recipe_items": {
                "type": "array",
                "items": {
                    "type": "object",
                    "properties": {
                        "name": {"type": "string"},
                        "servings": {"type": "integer"},
                        "ingredients": {
                            "type": "array",
                            "items": {
                                "type": "object",
                                "properties": {
                                    "name": {"type": "string"},
                                    "quantity": {"type": "number"},
                                    "unit": {"type": "string"},
                                },
                                "required": ["name", "quantity", "unit"],
                                "additionalProperties": False,
                            },
                        },
                        "steps": {
                            "type": "array",
                            "items": {"type": "string"},
                            "description": "Ordered cooking steps",
                        },
                    },
                    "required": ["name", "servings", "ingredients", "steps"],
                    "additionalProperties": False,
                },
            }
        },
        "required": ["recipe_items"],
        "additionalProperties": False,
    },
}

adjust_recipe_servings_json = {
    "name": "adjust_recipe_servings",
    "description": "Adjust the servings of a recipe in the recipes[] list so that it matches the desired number of servings",
    "parameters": {
        "type": "object",
        "properties": {
            "recipe_name": {"type": "string", "description": "The name of the recipe to adjust"},
            "desired_servings": {"type": "integer", "description": "The desired number of servings"}
        },
        "required": ["recipe_name", "desired_servings"],
        "additionalProperties": False
    }
}

adjust_all_recipes_servings_json = {
    "name": "adjust_all_recipes_servings",
    "description": "Scale every recipe in recipes[] to the same desired number of servings (meals) in one call.",
    "parameters": {
        "type": "object",
        "properties": {
            "desired_servings": {
                "type": "integer",
                "description": "Target servings per recipe (e.g. number of meals)",
            }
        },
        "required": ["desired_servings"],
        "additionalProperties": False,
    },
}

reconcile_recipes_with_pantry_json = {
    "name": "reconcile_recipes_with_pantry",
    "description": "Compare each recipe's ingredient quantities to the original pantry. Fills available_ingredients_used and additional_ingredients_required (each row includes recipe name). Call after scaling servings.",
    "parameters": {
        "type": "object",
        "properties": {
            "recipe_names": {
                "type": "array",
                "items": {"type": "string"},
                "description": "Optional: only these recipe names. Omit to reconcile all recipes in recipes[].",
            }
        },
        "required": [],
        "additionalProperties": False,
    },
}

In [31]:
tools = [
    {"type": "function", "function": set_available_ingredients_json},
    {"type": "function", "function": create_recipes_list_json},
    {"type": "function", "function": adjust_recipe_servings_json},
    {"type": "function", "function": adjust_all_recipes_servings_json},
    {"type": "function", "function": reconcile_recipes_with_pantry_json},
]

In [32]:
# Agent functions & loop

def handle_tool_calls(tool_calls):
    results = []
    for tool_call in tool_calls:
        tool_name = tool_call.function.name
        arguments = json.loads(tool_call.function.arguments)
        tool = globals().get(tool_name)
        if not tool:
            payload = {"error": f"Unknown tool: {tool_name}"}
        else:
            try:
                payload = tool(**arguments)
            except Exception as e:
                payload = {"error": str(e)}
        results.append(
            {
                "role": "tool",
                "content": json.dumps(payload, default=str),
                "tool_call_id": tool_call.id,
            }
        )
    return results

def loop(messages):
    done = False
    while not done:
        # show(f"\n\nmessages: {messages}\n\n")
        response = openai.chat.completions.create(model="gpt-5.2", messages=messages, tools=tools, reasoning_effort="none")
        # show(f"\n\nresponse: {response}\n\n")
        finish_reason = response.choices[0].finish_reason
        if finish_reason=="tool_calls":
            message = response.choices[0].message
            tool_calls = message.tool_calls
            results = handle_tool_calls(tool_calls)
            messages.append(message)
            messages.extend(results)
        else:
            done = True
    show(response.choices[0].message.content)

In [33]:
# Prompt

system_message = """
You are a smart cookbook agent. The user describes ingredients in free text and how many meals (servings) they need.

Tool order (always):
1) set_available_ingredients — Parse the user's pantry into items with name, quantity, and unit (e.g. flour 100 g). Use the same units the user gave.
2) create_recipes_list — Add exactly 3 recipes that use as much of the pantry as possible; ingredient names/units should match the pantry where possible.
3) adjust_all_recipes_servings — Scale all three recipes to the requested number of meals in one call (preferred). Or call adjust_recipe_servings three times.
4) reconcile_recipes_with_pantry — Run once with no arguments to fill used vs additional shopping list per recipe.

In your final reply, summarize the three recipes, steps, what the pantry covers, and what to buy extra (consistent with reconcile tool output).
Then give a conclusion : the most efficient recipe among the 3, given available ingredients.
If any quantity isn't provided, state a reasonable assumption in the steps.
Use Rich console markup only (no markdown code fences).
Do not ask the user questions; respond after tools complete.
"""

user_message = """
I have the following ingredients in my kitchen:
- 3 tomatoes
- 1 eggplant
- 2 eggs
- minsted beef (300g)
- olives (100g)
- 20g butter
And we are a family of 10 person, so I need to cook 10 meals.
"""

messages = [{"role": "system", "content": system_message}, {"role": "user", "content": user_message}]

In [34]:
loop(messages)

PANTRY SET: 6 items

CREATED RECIPES

Recipe #1: Beef & Eggplant Tomato-Olive Skillet (served with rice) - 4 servings
Recipe #2: Shakshuka-Style Eggs in Tomato & Olive Sauce (with bread) - 4 servings
Recipe #3: Greek-Style Stuffed Eggplant with Beef, Tomatoes & Olives (with pasta) - 4 servings

ADJUSTED SERVINGS FOR Beef & Eggplant Tomato-Olive Skillet (served with rice) from 4 to 10

ADJUSTED SERVINGS FOR Shakshuka-Style Eggs in Tomato & Olive Sauce (with bread) from 4 to 10

ADJUSTED SERVINGS FOR Greek-Style Stuffed Eggplant with Beef, Tomatoes & Olives (with pasta) from 4 to 10

RECONCILED PANTRY VS RECIPES

used entries: 14, additional needed entries: 38

Goal: 10 servings (meals) for each recipe, using your pantry: 3 tomatoes, 1 eggplant, 2 eggs, minced beef (300 g), 
olives (100 g), butter (20 g).

Recipe 1 — Beef & Eggplant Tomato-Olive Skillet (served with rice) (10 servings)
What you’ll do
1) Cook rice 750 g in salted water; keep warm.  
2) Dice eggplant 2.5 count (assumption: you’ll buy extra), lightly salt 10 min, then pat dry.  
3) Brown minced beef 750 g with onion 2.5 count in olive oil 5 tbsp.  
4) Add garlic 7.5 cloves + paprika 2.5 tsp for 30 seconds.  
5) Add eggplant; cook until softened.  
6) Add chopped tomatoes 7.5 count (splash of water if needed). Simmer to a sauce.  
7) Stir in olives 250 g; season with salt 2.5 tsp, black pepper 1.25 tsp. Serve over rice.  
Pantry covers (used): minced beef 300 g, eggplant 1 count, tomatoes 3 count, olives 100 g, butter 20 g  
Buy extra: minced beef 450 g; eggplant 1.5 count; tomatoes 4.5 count; olives 150 g; butter 30 g; rice 750 g; onion 
2.5 count; garlic 7.5 cloves; olive oil 5 tbsp; salt 2.5 tsp; black pepper 1.25 tsp; paprika 2.5 tsp

Recipe 2 — Shakshuka-Style Eggs in Tomato & Olive Sauce (with bread) (10 servings)
What you’ll do
1) Sauté onion 2.5 count in olive oil 2.5 tbsp + butter 50 g. Add garlic 5 cloves.  
2) Add chopped tomatoes 7.5 count + canned tomatoes 1000 g. Simmer until thick.  
3) Stir in olives 250 g. Season with salt 2.5 tsp, cumin 2.5 tsp, chili flakes 1.25 tsp.  
4) Make wells; crack in eggs 5 count (assumption: 1 egg per 2 servings here). Cover and cook until set.  
5) Serve with bread 1000 g.  
Pantry covers (used): tomatoes 3 count, olives 100 g, eggs 2 count, butter 20 g  
Buy extra: tomatoes 4.5 count; olives 150 g; eggs 3 count; butter 30 g; onion 2.5 count; garlic 5 cloves; canned 
tomatoes 1000 g; olive oil 2.5 tbsp; salt 2.5 tsp; cumin 2.5 tsp; chili flakes 1.25 tsp; bread 1000 g

Recipe 3 — Greek-Style Stuffed Eggplant with Beef, Tomatoes & Olives (with pasta) (10 servings)
What you’ll do
1) Heat oven to 200°C (assumption).  
2) Halve eggplant 2.5 count; scoop some flesh and chop it.  
3) Bake eggplant halves 15 min with a little olive oil 5 tbsp, salt, pepper.  
4) Cook filling: sauté onion 2.5 count in butter 50 g + a little oil; add garlic 5 cloves.  
5) Brown minced beef 750 g. Add chopped eggplant flesh + chopped tomatoes 7.5 count + oregano 2.5 tsp; simmer.  
6) Stir in olives 250 g; fill eggplants.  
7) Top with breadcrumbs 125 g + feta cheese 250 g; bake 15–20 min.  
8) Cook pasta 1000 g; serve alongside.  
Pantry covers (used): eggplant 1 count, minced beef 300 g, tomatoes 3 count, olives 100 g, butter 20 g  
Buy extra: eggplant 1.5 count; minced beef 450 g; tomatoes 4.5 count; olives 150 g; butter 30 g; onion 2.5 count; 
garlic 5 cloves; breadcrumbs 125 g; feta cheese 250 g; olive oil 5 tbsp; salt 2.5 tsp; black pepper 1.25 tsp; dried
oregano 2.5 tsp; pasta 1000 g

Conclusion — most efficient with your current pantry
Recipe 1 (Beef & Eggplant Tomato-Olive Skillet) is the most pantry-efficient: it uses all of your core items (beef,
eggplant, tomatoes, olives, butter) and only needs common add-ons (mainly rice + basic aromatics/spices), without 
requiring extra specialty toppings (like feta/breadcrumbs) or lots of canned tomatoes.